In [0]:
from pyspark.sql import functions as F

df_raw = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .option("pathGlobFilter", "*.pdf")
        .load("/Volumes/krishi_dhwani/bronze/icar_volume/")
)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS krishi_dhwani.bronze.checkpoints;

In [0]:
(
    df_raw.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", "/Volumes/krishi_dhwani/bronze/checkpoints/icar_raw")
        .trigger(once=True)   # VERY IMPORTANT (cost saving)
        .table("krishi_dhwani.bronze.icar_raw")
)

In [0]:
spark.sql("SELECT COUNT(*) FROM krishi_dhwani.bronze.icar_raw").show()

+--------+
|COUNT(*)|
+--------+
|       1|
+--------+



In [0]:
import random
from datetime import datetime, timedelta

prices = []
crops = ["Mustard", "Wheat", "Paddy", "Soybean", "Maize"]
mandis = ["Delhi", "Amritsar", "Jaipur", "Bhopal", "Lucknow"]

for i in range(30):  # last 30 days
    for crop in crops:
        prices.append({
            "date": (datetime.today() - timedelta(days=i)).strftime("%Y-%m-%d"),
            "crop": crop,
            "mandi": random.choice(mandis),
            "price_per_quintal": round(random.uniform(1500, 6000), 2),
            "unit": "quintal"
        })

spark.createDataFrame(prices).write.format("delta").mode("overwrite") \
    .saveAsTable("krishi_dhwani.bronze.market_prices_raw")

In [0]:
spark.table("krishi_dhwani.bronze.market_prices_raw").show(5)

+-------+----------+--------+-----------------+-------+
|   crop|      date|   mandi|price_per_quintal|   unit|
+-------+----------+--------+-----------------+-------+
|Mustard|2026-04-07|   Delhi|          4702.08|quintal|
|  Wheat|2026-04-07|  Bhopal|          3783.11|quintal|
|  Paddy|2026-04-07|Amritsar|          4248.51|quintal|
|Soybean|2026-04-07|  Jaipur|          5460.43|quintal|
|  Maize|2026-04-07|  Bhopal|          2202.42|quintal|
+-------+----------+--------+-----------------+-------+
only showing top 5 rows


In [0]:
weather = []
pincodes = ["110001", "141001", "302001", "462001", "226001"]

for pincode in pincodes:
    for day in range(7):
        weather.append({
            "pincode": pincode,
            "forecast_date": (datetime.today() + timedelta(days=day)).strftime("%Y-%m-%d"),
            "rainfall_mm": round(random.uniform(0, 40), 1),
            "temp_max_c": round(random.uniform(25, 42), 1),
            "temp_min_c": round(random.uniform(15, 28), 1),
            "humidity_pct": random.randint(40, 90),
            "condition": random.choice(["Clear", "Partly Cloudy", "Rain", "Thunderstorm"])
        })

spark.createDataFrame(weather).write.format("delta").mode("overwrite") \
    .saveAsTable("krishi_dhwani.bronze.weather_raw")

In [0]:
farmers = [
    {
        "farmer_id": "F001",
        "name": "Rajinder Singh",
        "pincode": "141001",
        "land_acres": 5.0,
        "soil_type": "Loamy",
        "ph": 6.8,
        "nitrogen_kg_ha": 180,
        "phosphorus_kg_ha": 45,
        "potassium_kg_ha": 220,
        "primary_crop": "Wheat",
        "state": "Punjab",
        "language": "punjabi"
    },
    {
        "farmer_id": "F002",
        "name": "Sunita Devi",
        "pincode": "302001",
        "land_acres": 2.5,
        "soil_type": "Sandy Loam",
        "ph": 7.2,
        "nitrogen_kg_ha": 120,
        "phosphorus_kg_ha": 30,
        "potassium_kg_ha": 150,
        "primary_crop": "Mustard",
        "state": "Rajasthan",
        "language": "hindi"
    }
]

spark.createDataFrame(farmers).write.format("delta").mode("overwrite") \
    .saveAsTable("krishi_dhwani.bronze.farmer_profiles_raw")